# 06: Exported classifier and exact soundness (gate G2)

Design decisions
- Tasks: BRFSS 2015 heart disease (task A) and CDC Diabetes (task B), the public tables from notebook 01. Label-independent splits with seed 42 (training portion 40%, split into dev-train 75% and dev-cal 25%; calibration split 1 30%; calibration split 2 15%; test 15%). Split codes are stored on Drive and their hashes committed. Only dev-train and dev-cal rows are used here; calibration and test rows stay unopened until notebook 05 freezes the configuration.
- Development models, trained on dev-train: XGBoost with 100 trees of depth 4 (plan default); the same XGBoost regularised with min_child_weight 50, gamma 2 and max_bin 32; LightGBM with 100 trees, at most 31 leaves and depth 6; random forest with 50 trees of depth 6. The threshold tau maximises sensitivity plus specificity on 5-fold out-of-fold scores of dev-train.
- Each model is exported to the exact format (stored thresholds, comparison rule, leaf values as exact floats) and scored by SCAX's own predictor in integer arithmetic. Exported and native decisions are compared on dev-train and dev-cal; any disagreement is listed with its distance from tau.
- Soundness procedures: the interval pre-check; the primary exact check (leaf branch-and-bound over grid cells, integer arithmetic); the second exact check (real thresholds and fractions, no shared data structure); Z3 on the real-input encoding; Z3 on the cell-literal encoding; brute-force enumeration. On every development model, 300 boxes built from dev-cal patients by freeing 3 to 18 random features are checked by the primary and second procedures, and the first 30 also by Z3 (20 s timeout). On three small ensembles over four coarse features, 400 random boxes per model and class are checked by all six, with brute force as ground truth.
- Gate G2: no disagreement between any two procedures that both return an answer, no disagreement between exported and native decisions beyond rounding at tau, and every unit test passes.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver', 'statsmodels'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scax import paths
from scax.utils import seed_everything, write_json
from scax.data import public, splits
from scax.models import train
from scax.models.export import Ensemble, export_model
from scax.models.predictor import ExactPredictor, native_scores
from scax.encode.interval_bound import IntervalBound
from scax.encode.cell_search import CellSearch
from scax.encode.real_search import RealSearch
from scax.encode.real_encoding import Z3RealEncoding
from scax.encode.z3_encoding import Z3CellEncoding
from scax.encode.bruteforce import brute_force_sound

SEED = 42
seed_everything(SEED)
paths.ensure_dirs()
MODEL_DIR = paths.DERIVED / 'models'; MODEL_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR = paths.DERIVED / 'splits'; SPLIT_DIR.mkdir(parents=True, exist_ok=True)
TASKS = ['brfss2015_heart', 'cdc_diabetes']
MODELS = {
    'xgboost': ('xgboost', {}),
    'xgboost_reg': ('xgboost', {'min_child_weight': 50, 'gamma': 2, 'max_bin': 32}),
    'lightgbm': ('lightgbm', {}),
    'sklearn_rf': ('sklearn_rf', {}),
}

## Label-independent splits

In [ ]:
DATA, SPLITS = {}, {}
for task in TASKS:
    X, y = public.LOADERS[task](paths.RAW)
    code = splits.make_splits(len(y), SEED)
    f = SPLIT_DIR / f'{task}.npy'
    if f.exists():
        assert np.array_equal(np.load(f), code), f'stored split for {task} differs'
    else:
        np.save(f, code)
    DATA[task], SPLITS[task] = (X, y), code
SPLIT_SUMMARY = {task: splits.summary(code) for task, code in SPLITS.items()}
print(pd.DataFrame(SPLIT_SUMMARY).T.to_string())
print('same respondents, same split for both tasks:', np.array_equal(SPLITS[TASKS[0]], SPLITS[TASKS[1]]))
write_json(f'{paths.RESULTS}/06_splits.json', {'seed': SEED, 'shares': splits.SHARES, 'dev_train_share': splits.DEV_TRAIN_SHARE,
                                              'splits': SPLIT_SUMMARY})

## Train, choose tau, export, compare with the native library

In [ ]:
ROWS, DISAGREE, MODEL_INFO = [], [], {}
for task in TASKS:
    X, y = DATA[task]
    Xa = X.to_numpy(dtype=float)
    tr, dc = splits.indices(SPLITS[task], 'dev_train'), splits.indices(SPLITS[task], 'dev_cal')
    for name, (family, params) in MODELS.items():
        t0 = time.time()
        oof = train.oof_scores(family, Xa[tr], y[tr], params, SEED)
        tau = train.youden_threshold(oof, y[tr])
        model = train.fit(family, Xa[tr], y[tr], params, SEED)
        ens = export_model(model, family, list(X.columns))
        ens.tau = tau
        ens.meta.update({'task': task, 'model': name, 'params': params, 'seed': SEED,
                         'tau_rule': 'Youden on 5-fold out-of-fold scores of dev-train'})
        ens.save(MODEL_DIR / f'{task}__{name}.json')
        pred = ExactPredictor(ens)
        g = pred.grid
        for split_name, idx in (('dev_train', tr), ('dev_cal', dc)):
            nat = native_scores(model, family, Xa[idx])
            exp_score = pred.score(Xa[idx])
            exp_pred = pred.predict(Xa[idx])
            nat_pred = (nat > tau).astype(np.int8)
            dis = np.flatnonzero(exp_pred != nat_pred)
            yy = y[idx]
            ROWS.append({'task': task, 'model': name, 'split': split_name, 'rows': len(idx), 'disagreements': len(dis),
                         'max_abs_score_difference': float(np.max(np.abs(exp_score - nat))),
                         'auroc_native': roc_auc_score(yy, nat), 'auroc_exported': roc_auc_score(yy, exp_score),
                         'sensitivity': float(exp_pred[yy == 1].mean()), 'specificity': float(1 - exp_pred[yy == 0].mean())})
            for i in dis:
                DISAGREE.append({'task': task, 'model': name, 'split': split_name, 'row': int(idx[i]),
                                 'exported_minus_tau': float(pred.margin_to_tau(Xa[idx[i]:idx[i] + 1])[0]),
                                 'native_minus_tau': float(nat[i] - tau)})
        MODEL_INFO[f'{task}__{name}'] = {'sha256': ens.hash(), 'family': family, 'params': params, 'tau': tau,
                                         'trees': ens.n_trees, 'active_features': int(len(g.active)),
                                         'thresholds': int(g.K.sum()), 'log10_family_size': g.family_size_log10(),
                                         'exact_scale_exponent': pred.E, 'seconds': time.time() - t0}
AGREE = pd.DataFrame(ROWS)
AGREE.to_csv(f'{paths.RESULTS}/06_exported_vs_native.csv', index=False)
print(AGREE.round(5).to_string(index=False))
print(pd.DataFrame(MODEL_INFO).T[['family', 'tau', 'active_features', 'thresholds', 'log10_family_size']].to_string())
print('disagreements between exported and native decisions:', len(DISAGREE))
if DISAGREE:
    print(pd.DataFrame(DISAGREE).to_string(index=False))

## Soundness procedures on the development models

In [ ]:
Z3_N = 30
rng = np.random.default_rng(SEED)
ENC_ROWS, ENC_FAIL = [], []
for task in TASKS:
    X, y = DATA[task]
    dc = splits.indices(SPLITS[task], 'dev_cal')
    sample = rng.choice(dc, 300, replace=False)
    for name in MODELS:
        ens = Ensemble.load(MODEL_DIR / f'{task}__{name}.json')
        pred = ExactPredictor(ens); g = pred.grid
        ib = IntervalBound(pred); cs = CellSearch(ib); rs = RealSearch(ens, g.thresholds)
        zr = Z3RealEncoding(ens, g.thresholds, timeout_ms=20000)
        C = g.cells(X.to_numpy(dtype=float)[sample]); yh = pred.predict_cells(C)
        res = {'task': task, 'model': name, 'boxes': len(C), 'sound': 0, 'primary_vs_second_disagree': 0,
               'primary_unknown': 0, 'second_unknown': 0, 'precheck_proved': 0, 'precheck_wrong': 0,
               'z3_checked': 0, 'z3_unknown': 0, 'z3_disagree': 0, 'primary_ms': 0.0, 'second_ms': 0.0, 'z3_ms': []}
        for i in range(len(C)):
            box = list(g.point_box(C[i]))
            free = rng.choice(g.active, size=int(rng.integers(3, min(18, len(g.active)) + 1)), replace=False)
            for j in free:
                box[j] = (0, int(g.K[j]))
            box = tuple(box); yb = int(yh[i])
            t0 = time.perf_counter(); a = cs.check(box, yb); res['primary_ms'] += 1000 * (time.perf_counter() - t0)
            t0 = time.perf_counter(); b = rs.check(box, yb); res['second_ms'] += 1000 * (time.perf_counter() - t0)
            res['primary_unknown'] += a == 'unknown'; res['second_unknown'] += b == 'unknown'
            if 'unknown' not in (a, b) and a != b:
                res['primary_vs_second_disagree'] += 1; ENC_FAIL.append((task, name, 'second', g.key(box), yb, a, b))
            res['sound'] += a == 'sound'
            if ib.sound(box, yb):
                res['precheck_proved'] += 1
                if a != 'sound':
                    res['precheck_wrong'] += 1; ENC_FAIL.append((task, name, 'precheck', g.key(box), yb, a, 'sound'))
            if i < Z3_N:
                t0 = time.perf_counter(); c = zr.check(box, yb); res['z3_ms'].append(1000 * (time.perf_counter() - t0))
                res['z3_checked'] += 1; res['z3_unknown'] += c == 'unknown'
                if c != 'unknown' and a != 'unknown' and c != a:
                    res['z3_disagree'] += 1; ENC_FAIL.append((task, name, 'z3', g.key(box), yb, a, c))
        res['primary_ms'] /= len(C); res['second_ms'] /= len(C)
        res['z3_ms_median'] = float(np.median(res.pop('z3_ms')))
        ENC_ROWS.append(res)
ENC = pd.DataFrame(ENC_ROWS)
print(ENC.round(2).to_string(index=False))

## Brute force on small ensembles

In [ ]:
SMALL_FEATURES = ['HighBP', 'HighChol', 'GenHlth', 'Age']
SMALL = {'xgboost': {'n_estimators': 10, 'max_depth': 3}, 'lightgbm': {'n_estimators': 10, 'num_leaves': 8, 'max_depth': 3},
         'sklearn_rf': {'n_estimators': 8, 'max_depth': 3}}
X, y = DATA['brfss2015_heart']
tr = splits.indices(SPLITS['brfss2015_heart'], 'dev_train')
Xs = X[SMALL_FEATURES].to_numpy(dtype=float)[tr]; ys = y[tr]
BF_ROWS = []
for family, params in SMALL.items():
    m = train.fit(family, Xs, ys, params, SEED)
    ens = export_model(m, family, SMALL_FEATURES)
    ens.tau = train.youden_threshold(native_scores(m, family, Xs), ys)
    pred = ExactPredictor(ens); g = pred.grid
    ib = IntervalBound(pred); cs = CellSearch(ib); rs = RealSearch(ens, g.thresholds)
    zr = Z3RealEncoding(ens, g.thresholds); zc = Z3CellEncoding(pred)
    out = {'family': family, 'thresholds': g.K.tolist(), 'checks': 0, 'sound': 0}
    for k in ('primary', 'second', 'z3_real', 'z3_cell', 'precheck_wrong'):
        out[k + '_errors'] = 0
    for _ in range(400):
        box = []
        for k in g.K:
            k = int(k); a = int(rng.integers(0, k + 1)); b = int(rng.integers(a, k + 1))
            box.append((0, k) if rng.random() < 0.3 else (a, b))
        box = tuple(box)
        for yb in (0, 1):
            truth = brute_force_sound(g, box, yb)
            out['checks'] += 1; out['sound'] += truth
            out['primary_errors'] += (cs.check(box, yb) == 'sound') != truth
            out['second_errors'] += (rs.check(box, yb) == 'sound') != truth
            out['z3_real_errors'] += (zr.check(box, yb) == 'sound') != truth
            out['z3_cell_errors'] += (zc.check(box, yb) == 'sound') != truth
            out['precheck_wrong_errors'] += ib.sound(box, yb) and not truth
    BF_ROWS.append(out)
BF = pd.DataFrame(BF_ROWS)
print(BF.to_string(index=False))

## Unit tests

In [ ]:
tests = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--color=no', '-p', 'no:cacheprovider', 'tests'],
                       capture_output=True, text=True, cwd=REPO_DIR)
print(tests.stdout[-3000:]); print(tests.stderr[-2000:])
TESTS_PASSED = tests.returncode == 0

## Gate G2

In [ ]:
far_disagreements = [d for d in DISAGREE if abs(d['native_minus_tau']) > 1e-5]
G2 = {
    'exported_vs_native_disagreements': len(DISAGREE),
    'exported_vs_native_disagreements_beyond_1e-5_of_tau': len(far_disagreements),
    'primary_vs_second_disagreements': int(ENC['primary_vs_second_disagree'].sum()),
    'precheck_wrong': int(ENC['precheck_wrong'].sum()),
    'z3_disagreements': int(ENC['z3_disagree'].sum()),
    'z3_unknown': int(ENC['z3_unknown'].sum()), 'z3_checked': int(ENC['z3_checked'].sum()),
    'brute_force_errors': int(BF[[c for c in BF.columns if c.endswith('_errors')]].to_numpy().sum()),
    'brute_force_checks': int(BF['checks'].sum()),
    'unit_tests_passed': bool(TESTS_PASSED),
}
G2['passed'] = bool(G2['exported_vs_native_disagreements_beyond_1e-5_of_tau'] == 0 and G2['primary_vs_second_disagreements'] == 0
                    and G2['precheck_wrong'] == 0 and G2['z3_disagreements'] == 0 and G2['brute_force_errors'] == 0
                    and TESTS_PASSED)
write_json(f'{paths.RESULTS}/06_encoding_tests.json', {'seed': SEED, 'gate_G2': G2, 'models': MODEL_INFO,
                                                       'development_boxes': ENC_ROWS, 'brute_force': BF_ROWS,
                                                       'exported_vs_native_disagreements': DISAGREE,
                                                       'failures': [list(map(str, f)) for f in ENC_FAIL]})
print(json.dumps(G2, indent=1))
assert G2['passed'], 'gate G2 failed: see failures in results/06_encoding_tests.json'

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "06: exported classifiers for tasks A and B (four development models), exported vs native, soundness procedures against each other and brute force, gate G2"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)